In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
import pandas as pd
import os
import shutil
from numba import njit
from google.colab import files

# ==========================================
# 1. シミュレーション・パラメータ設定
# ==========================================
N_LIST = [1000, 10000, 100000] # スケール不変性の確認
BURN_IN_STEPS = 100            # Phase 0: 自己組織化 (t=0~99)
ATTACK_STEPS = 100             # Phase 1: 病理発生 (t=100~199)
INTERVENTION_STEPS = 100       # Phase 2: アルゴリズム介入 (t=200~299)
TOTAL_STEPS = BURN_IN_STEPS + ATTACK_STEPS + INTERVENTION_STEPS
DT = 0.1

B_PAYLOAD = 0.9                # ハブへの偽ペイオフ
PENALTY_1D = 0.05              # 1次元介入による多孔性(Porosity)悪化係数
BOOST_2D = 5.0                 # 2次元介入(2-simplex注入)による修復力強化係数

OUTPUT_DIR = "simulation_results_intervention"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. Numbaによる高速SDE積分エンジン
# ==========================================
@njit
def sde_update_intervention(X, D, degrees, b_val, N, intervention_type,
                            dt=0.1, sigma=0.05, gamma=6.0, k_break=0.6, k_restore=0.1):

    total_degree = np.sum(degrees)
    mean_field = np.sum(X * degrees) / total_degree if total_degree > 0 else 0.0

    # 確率的認知ノイズ
    noise_X = np.random.normal(0, 1, N) * np.sqrt(dt)

    # コンフォーミティ(同調圧力)の更新
    dX = (mean_field + b_val - X) * dt + sigma * noise_X
    X_new = np.minimum(np.maximum(X + dX, 0.0), 1.0)

    # 介入タイプによる修復キャパシティの動的変動
    repair_capacity = np.exp(-gamma * D)

    if intervention_type == 2:  # 高次元(2-simplex)介入
        repair_capacity = repair_capacity * BOOST_2D

    dD = (k_break * X_new * (1.0 - D) - k_restore * (1.0 - X_new) * D * repair_capacity) * dt
    D_new = np.minimum(np.maximum(D + dD, 0.0), 1.0)

    if intervention_type == 1:  # 1次元(エッジ)介入によるペナルティ(多孔性の悪化)
        D_new = np.minimum(D_new + PENALTY_1D * (1.0 - D_new) * dt, 1.0)

    return X_new, D_new

# ==========================================
# 3. メインシミュレーションループ
# ==========================================
ks_results = []
defect_results = []

for N in N_LIST:
    print(f"--- Simulating N = {N} ---")

    np.random.seed(42)
    degrees = np.random.pareto(2.5, N) + 1.0
    hub_idx = np.argmax(degrees)

    scenarios = ["No_Intervention", "1D_Edge_Injection", "2D_Simplex_Injection"]

    for scenario in scenarios:
        np.random.seed(42)
        X = np.random.uniform(0.0, 0.2, N)
        D = np.random.uniform(0.0, 0.1, N)
        b_val = np.zeros(N)
        X_history = []

        for step in range(TOTAL_STEPS):
            intervention_flag = 0

            # フェーズ制御
            if step >= BURN_IN_STEPS and step < BURN_IN_STEPS + ATTACK_STEPS:
                b_val[hub_idx] = B_PAYLOAD # 攻撃フェーズ
            elif step >= BURN_IN_STEPS + ATTACK_STEPS:
                b_val[hub_idx] = 0.0       # 攻撃終了・介入フェーズ開始
                if scenario == "1D_Edge_Injection":
                    intervention_flag = 1
                elif scenario == "2D_Simplex_Injection":
                    intervention_flag = 2

            # KS統計量記録 (Phase 0)
            if step < BURN_IN_STEPS and scenario == "No_Intervention" and step % 10 == 0:
                X_history.append(X.copy())
                if len(X_history) > 1:
                    stat, _ = ks_2samp(X_history[-1], X_history[-2])
                    ks_results.append({"N": N, "Step": step, "KS_Statistic": stat})

            # SDE更新
            X, D = sde_update_intervention(X, D, degrees, b_val, N, intervention_flag, dt=DT)

            # データ記録
            if step % 5 == 0:
                defect_results.append({
                    "N": N,
                    "Scenario": scenario,
                    "Step": step,
                    "Defect_D": np.mean(D)
                })

# ==========================================
# 4. データ保存と可視化 (CSV & グラフ)
# ==========================================
df_ks = pd.DataFrame(ks_results)
df_defect = pd.DataFrame(defect_results)

df_ks.to_csv(f"{OUTPUT_DIR}/ks_convergence_intervention.csv", index=False)
df_defect.to_csv(f"{OUTPUT_DIR}/intervention_dynamics.csv", index=False)

# グラフ1: KS統計量
plt.figure(figsize=(8, 5))
for N in N_LIST:
    subset = df_ks[df_ks["N"] == N]
    plt.plot(subset["Step"], subset["KS_Statistic"], marker='o', label=f"N={N}")
plt.title("Phase 0: Ergodic Self-Organization (KS Convergence)")
plt.xlabel("Burn-in Steps")
plt.ylabel("KS Statistic ($D_{KS}$)")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/Fig1_KS_Convergence.png", dpi=300)
plt.close()

# グラフ2: 介入シナリオの比較 (N=100000)
plt.figure(figsize=(10, 6))
N_max = N_LIST[-1]
sub_no = df_defect[(df_defect["N"] == N_max) & (df_defect["Scenario"] == "No_Intervention")]
sub_1d = df_defect[(df_defect["N"] == N_max) & (df_defect["Scenario"] == "1D_Edge_Injection")]
sub_2d = df_defect[(df_defect["N"] == N_max) & (df_defect["Scenario"] == "2D_Simplex_Injection")]

plt.plot(sub_no["Step"], sub_no["Defect_D"], color='black', linestyle=':', label="Baseline (Hysteresis)")
plt.plot(sub_1d["Step"], sub_1d["Defect_D"], color='red', linestyle='--', label="1D Intervention (Edge Injection)")
plt.plot(sub_2d["Step"], sub_2d["Defect_D"], color='blue', linewidth=2, label="2D Intervention (2-simplex Injection)")

plt.axvline(x=BURN_IN_STEPS, color='gray', linestyle='-', alpha=0.5, label="Attack Start")
plt.axvline(x=BURN_IN_STEPS + ATTACK_STEPS, color='green', linestyle='-', alpha=0.5, label="Intervention Start")
plt.title(f"High-Dimensional Algorithm Intervention vs Topological Porosity (N={N_max})")
plt.xlabel("Time Step")
# 修正箇所: $\beta_1$ を正しく LaTeX フォーマット ($ $) で囲む
plt.ylabel("Systemic Topological Defect ($\\beta_1$)")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/Fig2_Intervention_Comparison.png", dpi=300)
plt.close()

# グラフ3: スケール不変性の確認 (最終欠陥量)
plt.figure(figsize=(8, 5))
final_step = TOTAL_STEPS - 5
for scenario, color, marker in zip(scenarios, ['black', 'red', 'blue'], ['o', 'x', 's']):
    sub = df_defect[(df_defect["Step"] == final_step) & (df_defect["Scenario"] == scenario)]
    plt.plot(sub["N"], sub["Defect_D"], color=color, marker=marker, label=scenario.replace('_', ' '))

plt.xscale('log')
plt.title("Scale-Invariant Efficacy of High-Dimensional Interventions")
plt.xlabel("Network Size N (Log Scale)")
plt.ylabel("Terminal Defect ($D$)")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/Fig3_Scale_Invariance.png", dpi=300)
plt.close()

# ==========================================
# 5. ZIP圧縮と自動ダウンロード
# ==========================================
shutil.make_archive("Simulation_Results_Hypothesis3", 'zip', OUTPUT_DIR)
print("シミュレーション完了。結果をZIP圧縮してダウンロードします...")
files.download("Simulation_Results_Hypothesis3.zip")

--- Simulating N = 1000 ---
--- Simulating N = 10000 ---
--- Simulating N = 100000 ---
シミュレーション完了。結果をZIP圧縮してダウンロードします...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>